gg

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from models.KAN import KAN

In [ ]:
x_loaded = torch.load("../data/x_data_hard.pt", weights_only=True)
y_loaded = torch.load("../data/y_data_hard.pt", weights_only=True)

x_train, x_test, y_train, y_test = train_test_split(
    x_loaded, y_loaded, test_size=0.2, random_state=42
)

print(f"Data shape: train {x_train.shape}, test {x_test.shape}")

Data shape: train torch.Size([4000, 2]), test torch.Size([1000, 2])


In [ ]:

def train_model(model, x_train, y_train, x_test, y_test, epochs=1000, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses, test_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(x_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()
        
        # eval
        model.eval()
        with torch.no_grad():
            test_pred = model(x_test)
            test_loss = criterion(test_pred, y_test).item()
        
        train_losses.append(loss.item())
        test_losses.append(test_loss)
        
        if epoch % 50 == 0:
            print(f"Epoch {epoch:3d} | Train MSE: {loss.item():.6f} | Test MSE: {test_loss:.6f}")
    
    return train_losses, test_losses


In [ ]:
###

# Note: This is not the training run used in the newsletter.

###

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_train, y_train = x_train.to(device), y_train.to(device)
x_test, y_test = x_test.to(device), y_test.to(device)

import os

MODEL_DIR = os.path.join("..", "src", "models", "saved") 
os.makedirs(MODEL_DIR, exist_ok=True)

KAN_PATH = os.path.join(MODEL_DIR, "kan.pth")
KAN_LOSSES_PATH = os.path.join(MODEL_DIR, "kan_losses.pt")


    
print("\n=== Training KAN ===")
kan = KAN(in_dim=2, out_dim=1, hidden_per_uni=8).to(device)
kan_train_loss, kan_test_loss = train_model(kan, x_train, y_train, x_test, y_test)

# Save models and losses
torch.save(kan.state_dict(), KAN_PATH)
torch.save({'train': kan_train_loss, 'test': kan_test_loss}, KAN_LOSSES_PATH)
print("Models and losses saved to disk!")



=== Training KAN ===
Epoch   0 | Train MSE: 2.544432 | Test MSE: 1.571926
Epoch  50 | Train MSE: 0.476069 | Test MSE: 0.486681
Epoch 100 | Train MSE: 0.470837 | Test MSE: 0.475642
Epoch 150 | Train MSE: 0.446493 | Test MSE: 0.443622
Epoch 200 | Train MSE: 0.357286 | Test MSE: 0.370369
Epoch 250 | Train MSE: 0.340424 | Test MSE: 0.356457
Epoch 300 | Train MSE: 0.306628 | Test MSE: 0.317217
Epoch 350 | Train MSE: 0.250907 | Test MSE: 0.263660
Epoch 400 | Train MSE: 0.161763 | Test MSE: 0.153359
Epoch 450 | Train MSE: 0.026039 | Test MSE: 0.028049
Epoch 500 | Train MSE: 0.017681 | Test MSE: 0.018581
Epoch 550 | Train MSE: 0.014003 | Test MSE: 0.014436
Epoch 600 | Train MSE: 0.011958 | Test MSE: 0.012292
Epoch 650 | Train MSE: 0.010198 | Test MSE: 0.010683
Epoch 700 | Train MSE: 0.009178 | Test MSE: 0.009768
Epoch 750 | Train MSE: 0.008845 | Test MSE: 0.009188
Epoch 800 | Train MSE: 0.007792 | Test MSE: 0.008307
Epoch 850 | Train MSE: 0.007346 | Test MSE: 0.007874
Epoch 900 | Train MSE: 0